# Практическая Работа - Основы Рекомендательных Систем

In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

data = { 
    'Film_ID': [1, 2, 3, 4, 5], 
    'Title': ['Die Hard', 'The Martian', 'Pulp Fiction', 'Guardians of the Galaxy', 'La La Land'], 
    'Action': [1, 0, 0, 1, 0], 
    'Comedy': [0, 0, 0, 1, 0], 
    'Drama': [0, 0, 1, 0, 1], 
    'Sci-Fi': [0, 1, 0, 1, 0] 
} 
movies_df = pd.DataFrame(data).set_index('Film_ID')

feature_matrix = movies_df[['Action', 'Comedy', 'Drama', 'Sci-Fi']] 
print("Матрица Признаков:\n", feature_matrix)

cosine_sim_matrix = cosine_similarity(feature_matrix) 
cosine_sim_df = pd.DataFrame(cosine_sim_matrix, index=movies_df['Title'], columns=movies_df['Title']) 
print("\nМатрица Косинусного Сходства:\n", cosine_sim_df)

def get_recommendations(title, cosine_sim_df, movies_df, top_n=2): 
    sim_scores = cosine_sim_df[title]
    sim_scores = sim_scores.sort_values(ascending=False)
    top_recommendations = sim_scores[1:top_n+1]
    return top_recommendations.index.tolist()

print(f"\nРекомендации для 'Die Hard': {get_recommendations('Die Hard', cosine_sim_df, movies_df, top_n=2)}")
print(f"Рекомендации для 'Pulp Fiction': {get_recommendations('Pulp Fiction', cosine_sim_df, movies_df, top_n=2)}")

Матрица Признаков:
          Action  Comedy  Drama  Sci-Fi
Film_ID                               
1             1       0      0       0
2             0       0      0       1
3             0       0      1       0
4             1       1      0       1
5             0       0      1       0

Матрица Косинусного Сходства:
 Title                    Die Hard  The Martian  Pulp Fiction  \
Title                                                          
Die Hard                  1.00000      0.00000           0.0   
The Martian               0.00000      1.00000           0.0   
Pulp Fiction              0.00000      0.00000           1.0   
Guardians of the Galaxy   0.57735      0.57735           0.0   
La La Land                0.00000      0.00000           1.0   

Title                    Guardians of the Galaxy  La La Land  
Title                                                         
Die Hard                                 0.57735         0.0  
The Martian                          

### Ответы на вопросы для отчета (Задание 1.3)

**1. Какое сходство система рассчитала между фильмами “The Martian” и “Guardians of the Galaxy”?**
Система рассчитала косинусное сходство, равное примерно **0.577** (или $1/\sqrt{3}$). 
**Объяснение:** Вектор признаков "The Martian" — `[0, 0, 0, 1]` (только Sci-Fi), а "Guardians" — `[1, 1, 0, 1]` (Action, Comedy, Sci-Fi). У них есть ровно один общий жанр, поэтому сходство не равно нулю. Но оно не равно и единице, потому что у "Guardians" есть дополнительные жанры, которых нет у "The Martian", из-за чего угол между их векторами в многомерном пространстве не является нулевым.

**2. Какой главный недостаток Content-Based фильтрации виден на примере “La La Land”?**
Если посмотреть на матрицу сходства, для "La La Land" `[0, 0, 1, 0]` (Drama) система с вероятностью 1.0 порекомендует "Pulp Fiction" `[0, 0, 1, 0]` (Drama). 
Недостаток - Это демонстрирует проблему слишком широких метаданных. Оба фильма относятся к жанру драма, но сюжетно, стилистически и по настроению они совершенно разные. Контентная фильтрация не понимает суть фильма, она видит только совпадение тега "Drama", что приводит к примитивным и иногда нелогичным рекомендациям.

In [2]:
from surprise import Dataset
from surprise.model_selection import train_test_split
from surprise import KNNWithMeans
from surprise import accuracy

data = Dataset.load_builtin('ml-100k')
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {trainset.n_ratings} оценок")
print(f"Размер тестовой выборки: {len(testset)} оценок")

sim_options = { 
    'name': 'pearson', 
    'user_based': True 
}

algo = KNNWithMeans(k=40, sim_options=sim_options)
algo.fit(trainset)
print("\nМодель User-Based CF (Пирсон) обучена.")

predictions = algo.test(testset)
rmse = accuracy.rmse(predictions, verbose=False)
print(f"\nRMSE на тестовой выборке: {rmse:.4f}")

user_id = '196' 
item_id = '302' 

prediction = algo.predict(user_id, item_id, verbose=False)
print(f"\nПредсказанная оценка для Пользователя {user_id} (Фильм {item_id}): {prediction.est:.3f}")

Dataset ml-100k could not be found. Do you want to download it? [Y/n] 

 Y


Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to C:\Users\Bigon/.surprise_data/ml-100k
Размер обучающей выборки: 80000 оценок
Размер тестовой выборки: 20000 оценок
Computing the pearson similarity matrix...
Done computing similarity matrix.

Модель User-Based CF (Пирсон) обучена.

RMSE на тестовой выборке: 0.9492

Предсказанная оценка для Пользователя 196 (Фильм 302): 4.175


### Ответы на вопросы для отчета (Задание 2.3)

**1. Что означает полученное значение RMSE?**
RMSE - это корень из среднего квадратичного отклонения предсказаний от реальных оценок. Простыми словами, это средняя ошибка нашей модели в звездах. Например, если RMSE = 0.95, это значит, что в среднем мы ошибаемся при предсказании оценки пользователя на 0.95 звезды.
Если бы RMSE было равно 0.5: Это означало бы, что модель стала значительно точнее. В среднем мы бы ошибались всего на половину звезды, что является очень высоким показателем точности для рекомендательных систем.

**2. Как изменится RMSE, если заменить 'pearson' на 'cosine'? Почему Пирсон лучше?**
При замене на 'cosine' значение RMSE, скорее всего, увеличится.
Объяснение - Косинусное сходство измеряет только угол между векторами оценок, игнорируя их масштаб. Но у пользователей разные шкалы оценок: один пользователь - критик и ставит всем 3 и 4, а другой - фанат и всем ставит 5. Корреляция Пирсона центрирует данные, благодаря чему она сравнивает не абсолютные значения, а отклонения от привычного среднего. Поэтому Пирсон учитывает личную базовую линию пользователя и дает гораздо более точные результаты в User-Based CF.

### Итоговый вывод: Какой подход перспективнее для крупного онлайн-кинотеатра?

Для крупного онлайн-кинотеатра Коллаборативная фильтрация является гораздо более перспективным и мощным подходом.

Почему CF выигрывает:
1. Обнаружение скрытых паттернов: CF находит связи, которые невозможно описать метаданными. Например, алгоритм может "понять", что люди, которые пересматривают серьезные исторические драмы, также любят специфические японские аниме, просто потому что у этой группы пользователей схожее поведение. Контентная фильтрация никогда такого не поймет, если мы вручную не зададим эти странные теги.
2. Независимость от качества описания: Content-Based полностью зависит от того, насколько качественно и подробно редакция размечает фильмы. Если в карточке фильма не указан жанр киберпанк, система его не найдет. CF нужны только факты: кто что посмотрел и какие оценки поставил.
3. Персонализация на уровне вкусов: CF группирует людей по реальному сходству их вкусов, что всегда работает лучше, чем поиск по общим тегам.

Единственный классический минус CF - проблема холодного старта. Но на масштабах крупного кинотеатра с миллионами просмотров эта проблема легко решается гибридными подходами.